In [15]:
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/91.0.4472.124 Safari/537.36"
    )
}

def clean_text(cell):
    text = unicodedata.normalize("NFKD", cell.get_text(" ", strip=True))
    text = re.sub(r"\[[^\]]*\]", "", text)   # remove [references]
    text = re.sub(r"\s+", " ", text).strip()
    return text

def date_time(table_cell):
    texts = [t.strip() for t in table_cell.stripped_strings if t.strip()]
    date = texts[0].replace(",", "") if len(texts) > 0 else None
    time = texts[1] if len(texts) > 1 else None
    return date, time

def booster_version(table_cell):
    texts = [t.strip() for t in table_cell.stripped_strings if t.strip()]
    if len(texts) >= 2:
        out = "".join(texts[::2]).strip()
        out = re.sub(r"\[[^\]]*\]", "", out).strip()
        return out
    return clean_text(table_cell)

def landing_status(table_cell):
    texts = [t.strip() for t in table_cell.stripped_strings if t.strip()]
    return texts[0] if texts else None

def get_mass(table_cell):
    text = clean_text(table_cell)
    match = re.search(r"[\d,]+ ?kg", text)
    return match.group(0) if match else None

def extract_column_from_header(row):
    text = unicodedata.normalize("NFKD", row.get_text(" ", strip=True))
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text and not text.isdigit():
        return text
    return None

# TASK 1: Request the page
response = requests.get(static_url, headers=headers, timeout=30)
response.raise_for_status()

# Create BeautifulSoup object
soup = BeautifulSoup(response.text, "html.parser")

# Verify title
print(soup.title)

# TASK 2: Find tables and extract headers
html_tables = soup.select("table.wikitable.plainrowheaders.collapsible")
print("Number of target tables found:", len(html_tables))

column_names = []
if len(html_tables) >= 3:
    first_launch_table = html_tables[2]
    for th in first_launch_table.find_all("th"):
        column_name = extract_column_from_header(th)
        if column_name is not None and len(column_name) > 0:
            column_names.append(column_name)

print("Extracted headers:")
print(column_names)

# TASK 3: Parse launch tables into a dataframe
launch_dict = {
    "Flight No.": [],
    "Date": [],
    "Time": [],
    "Version Booster": [],
    "Launch site": [],
    "Payload": [],
    "Payload mass": [],
    "Orbit": [],
    "Customer": [],
    "Launch outcome": [],
    "Booster landing": []
}

for table in html_tables:
    for row in table.find_all("tr"):
        header = row.find("th")
        if header is None:
            continue

        flight_number = header.get_text(" ", strip=True)
        if not flight_number.isdigit():
            continue

        cells = row.find_all("td")
        if len(cells) < 9:
            continue

        date, time = date_time(cells[0])

        launch_dict["Flight No."].append(flight_number)
        launch_dict["Date"].append(date)
        launch_dict["Time"].append(time)
        launch_dict["Version Booster"].append(booster_version(cells[1]))
        launch_dict["Launch site"].append(clean_text(cells[2]))
        launch_dict["Payload"].append(clean_text(cells[3]))
        launch_dict["Payload mass"].append(get_mass(cells[4]))
        launch_dict["Orbit"].append(clean_text(cells[5]))
        launch_dict["Customer"].append(clean_text(cells[6]))
        launch_dict["Launch outcome"].append(clean_text(cells[7]))
        launch_dict["Booster landing"].append(landing_status(cells[8]))

df = pd.DataFrame(launch_dict)

print(df.shape)
print(df.head())

# Export to CSV
df.to_csv("spacex_web_scraped.csv", index=False)
print("Saved as spacex_web_scraped.csv")

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>
Number of target tables found: 9
Extracted headers:
['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']
(121, 11)
  Flight No.             Date   Time   Version Booster     Launch site  \
0          1      4 June 2010  18:45  F9 v1.07B0003.18  CCAFS , SLC-40   
1          2  8 December 2010  15:43  F9 v1.07B0004.18  CCAFS , SLC-40   
2          3      22 May 2012  07:44  F9 v1.07B0005.18  CCAFS , SLC-40   
3          4   8 October 2012  00:35  F9 v1.07B0006.18  CCAFS , SLC-40   
4          5     1 March 2013  15:10  F9 v1.07B0007.18  CCAFS , SLC-40   

                                Payload Payload mass        Orbit  \
0  Dragon Spacecraft Qualification Unit         None          LEO   
1   Dragon demo flight C1 (Dragon C101)         None  LEO ( ISS )   
2  Dragon demo flight C2+ (Dragon C102)       525 kg  LE